# Attention Metrics and Asset Price Dynamics in Financial Time Series

This project analyzes the relationship between investor attention, proxied by Google Trends data, and financial market behavior. Specifically, it examines whether attention signals and shocks contain predictive information about future returns and volatility across selected assets.

**Author:** Atharva S  
**Date:** April 22 2026

## Imports

In [231]:
import time
import numpy as np
import pandas as pd
import yfinance as yf
from pytrends.request import TrendReq
import matplotlib.pyplot as plt
import statsmodels.api as sm
import os
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

## 2 & 3: Fetching Data and Data Processing

### Price/ETF data

In [37]:
tickers = ['SPY','BTC-USD','TSLA']
asset_data = yf.download(tickers, start = '2021-01-01')
asset_data.columns = ['_'.join(col)for col in asset_data.columns]
asset_data = asset_data.dropna()  # remove weekends

# create features
for ticker in tickers:
    asset_data[ticker+'_log_ret'] = np.log( asset_data['Close_'+ticker] / asset_data['Close_'+ticker].shift(1) )
    asset_data[ticker+'_log_ret_t+1'] = asset_data[ticker+'_log_ret'].shift(-1)
    asset_data[ticker+'_vol_10'] = asset_data[ticker+'_log_ret'].rolling(window=10).std()
    asset_data[ticker+'_vol_10_t+1'] = asset_data[ticker+'_vol_10'].shift(-1)
    asset_data[ticker+'_vol_change_t+1'] = asset_data[ticker+'_vol_10'].diff().shift(-1)
    asset_data[ticker+'_shock_10'] = asset_data[ticker+'_log_ret'] / asset_data[ticker+'_vol_10']
    asset_data[ticker+'_shock_10_t+1'] = asset_data[ticker+'_shock_10'].shift(-1)
    asset_data = asset_data.drop(columns=[i+'_'+ticker for i in ['Open','Close','High','Low','Volume']])  # remove unnecessary columns

asset_data = asset_data.dropna()
asset_data.to_csv('asset_data.csv')  # save to csv for reuse
asset_data

[*********************100%***********************]  3 of 3 completed


,SPY_log_ret,SPY_log_ret_t+1,SPY_vol_10,SPY_vol_10_t+1,SPY_vol_change_t+1,SPY_shock_10,SPY_shock_10_t+1,BTC-USD_log_ret,BTC-USD_log_ret_t+1,BTC-USD_vol_10,...,BTC-USD_vol_change_t+1,BTC-USD_shock_10,BTC-USD_shock_10_t+1,TSLA_log_ret,TSLA_log_ret_t+1,TSLA_vol_10,TSLA_vol_10_t+1,TSLA_vol_change_t+1,TSLA_shock_10,TSLA_shock_10_t+1
Date,,,,,,,,,,,,,,,,,,,,,
2021-01-19,0.007821,0.013743,0.007015,0.007772,0.000757,1.114935,1.768334,-0.020731,-0.014579,0.075489,...,-0.001703,-0.274621,-0.197589,0.022015,0.006962,0.047317,0.047323,0.000006,0.465273,0.147110
2021-01-20,0.013743,0.000912,0.007772,0.007746,-0.000026,1.768334,0.117688,-0.014579,-0.142528,0.073785,...,0.007829,-0.197589,-1.746364,0.006962,-0.006441,0.047323,0.047491,0.000168,0.147110,-0.135624
2021-01-21,0.000912,-0.003546,0.007746,0.006707,-0.001039,0.117688,-0.528676,-0.142528,0.068333,0.081614,...,0.000170,-1.746364,0.835536,-0.006441,0.001951,0.047491,0.041582,-0.005909,-0.135624,0.046914
2021-01-22,-0.003546,0.003936,0.006707,0.006593,-0.000114,-0.528676,0.597026,0.068333,-0.019562,0.081784,...,-0.002157,0.835536,-0.245665,0.001951,0.039555,0.041582,0.035845,-0.005736,0.046914,1.103501
2021-01-25,0.003936,-0.001562,0.006593,0.006116,-0.000477,0.597026,-0.255431,-0.019562,0.006266,0.079627,...,-0.010618,-0.245665,0.090806,0.039555,0.002597,0.035845,0.021645,-0.014200,1.103501,0.119960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-15,0.007860,0.002454,0.007517,0.007673,0.000156,1.045692,0.319860,0.008370,0.004629,0.018932,...,-0.000342,0.442093,0.248991,0.073431,-0.007812,0.035053,0.034516,-0.000537,2.094864,-0.226335
2026-04-16,0.002454,0.012013,0.007673,0.007520,-0.000153,0.319860,1.597582,0.004629,0.025937,0.018590,...,-0.002189,0.248991,1.581502,-0.007812,0.029691,0.034516,0.028728,-0.005788,-0.226335,1.033524
2026-04-17,0.012013,-0.002002,0.007520,0.008118,0.000598,1.597582,-0.246586,0.025937,-0.016397,0.016400,...,0.001655,1.581502,-0.908166,0.029691,-0.020477,0.028728,0.028567,-0.000161,1.033524,-0.716793


### Attention Data
Note: may take 20-30 minutes

In [2]:
pytrends = TrendReq()
SPY_kws = ['stock market', 'stock market crash','inflation','interest rates','recession','oil prices','crude oil','Fed rate hike','economy']
BTC_kws = ['bitcoin price','crypto crash','crypto','cryptocurrency','buy bitcoin','bitcoin ETF','inflation hedge','risk off','altcoin','memecoin', 'bitcoin mining']
TSLA_kws = ['tesla stock','elon musk','self driving car','electric vehicles','AI stocks','tesla crash','battery technology','lithium price']
kws = SPY_kws + BTC_kws + TSLA_kws

start_date = pd.to_datetime('2021-01-01')
end_date = pd.to_datetime('2026-04-19')
window_months = 6
overlap_months = 2
attention_data = None

current_start = start_date
current_end = current_start

while current_end < end_date:

    current_end = current_start+pd.DateOffset(months=window_months)

    if current_end > end_date:
        current_end = end_date
    
    timeframe = f'{current_start.date()} {current_end.date()}'
    print(f'\nFetching window {timeframe}')
    window_df = pd.DataFrame()


    for kw in kws:
        print(f'\rFetching {kw:<30}',end='',flush=True)

        fetched = False
        attempts = 0

        while (not fetched) and (attempts < 5):
            try:
                attempts += 1
                pytrends.build_payload(
                    kw_list = [kw],
                    timeframe = timeframe
                )
                data = pytrends.interest_over_time()
                fetched = True
            except Exception as e:
                print(f'Attempt {attempts} failed: {e}')
                time.sleep(60)  # wait 60s for rate limit

        data = data.drop(columns=['isPartial'], errors='ignore')  # remove unnecessary column
        window_df[kw] = data[kw]

    # Scale values to match previous window

    if attention_data is None:
        attention_data = window_df.copy()
    else:
        # get overlap index
        overlap = attention_data.index.intersection(window_df.index)

        # calculate scale factors
        scale_factors = {}
        for kw in kws:
            old_vals = attention_data.loc[overlap, kw]
            new_vals = window_df.loc[overlap, kw]

            scale = old_vals.mean() / new_vals.mean()

            scale_factors[kw]=scale

        # scale each kw
        for kw in kws:
            window_df[kw] = window_df[kw] * scale_factors[kw]
    
    # get non-overlapping part
    window_df = window_df.loc[~window_df.index.isin(attention_data.index)]
    attention_data = pd.concat([attention_data,window_df])

    current_start = current_end - pd.DateOffset(months=overlap_months)

attention_data.to_csv('raw_attention_data.csv')
attention_data


Fetching window 2021-01-01 2021-07-01
Fetching stock market                  Attempt 1 failed: The request failed: Google returned a response with code 429
Fetching tesla crash                   Attempt 1 failed: The request failed: Google returned a response with code 429
Fetching lithium price                 
Fetching window 2021-05-01 2021-11-01
Fetching lithium price                 
Fetching window 2021-09-01 2022-03-01
Fetching economy                       Attempt 1 failed: The request failed: Google returned a response with code 429
Fetching buy bitcoin                   Attempt 1 failed: The request failed: Google returned a response with code 429
Fetching elon musk                     Attempt 1 failed: The request failed: Google returned a response with code 429
Fetching lithium price                 
Fetching window 2022-01-01 2022-07-01
Fetching stock market                  

C:\Users\athar\AppData\Local\Temp\ipykernel_31568\3041995569.py:64: RuntimeWarning: invalid value encountered in scalar divide
  scale = old_vals.mean() / new_vals.mean()


Fetching stock market crash            Attempt 1 failed: The request failed: Google returned a response with code 429
Fetching buy bitcoin                   Attempt 1 failed: The request failed: Google returned a response with code 429
Fetching bitcoin ETF                   Attempt 1 failed: The request failed: Google returned a response with code 429
Fetching tesla stock                   Attempt 1 failed: The request failed: Google returned a response with code 429
Fetching lithium price                 
Fetching window 2022-05-01 2022-11-01
Fetching lithium price                 
Fetching window 2022-09-01 2023-03-01
Fetching lithium price                 
Fetching window 2023-01-01 2023-07-01
Fetching stock market                  

C:\Users\athar\AppData\Local\Temp\ipykernel_31568\3041995569.py:64: RuntimeWarning: invalid value encountered in scalar divide
  scale = old_vals.mean() / new_vals.mean()


Fetching lithium price                 
Fetching window 2023-05-01 2023-11-01
Fetching lithium price                 
Fetching window 2023-09-01 2024-03-01
Fetching lithium price                 
Fetching window 2024-01-01 2024-07-01
Fetching lithium price                 
Fetching window 2024-05-01 2024-11-01
Fetching lithium price                 
Fetching window 2024-09-01 2025-03-01
Fetching bitcoin price                 Attempt 1 failed: The request failed: Google returned a response with code 429
Fetching AI stocks                     Attempt 1 failed: The request failed: Google returned a response with code 429
Fetching lithium price                 
Fetching window 2025-01-01 2025-07-01
Fetching oil prices                    Attempt 1 failed: The request failed: Google returned a response with code 429
Fetching lithium price                 
Fetching window 2025-05-01 2025-11-01
Fetching oil prices                    Attempt 1 failed: The request failed: Google returned a respo

,stock market,stock market crash,inflation,interest rates,recession,oil prices,crude oil,Fed rate hike,economy,bitcoin price,...,memecoin,bitcoin mining,tesla stock,elon musk,self driving car,electric vehicles,AI stocks,tesla crash,battery technology,lithium price
date,,,,,,,,,,,,,,,,,,,,,
2021-01-01,23.000000,21.000000,24.000000,73.000000,46.000000,43.000000,27.000000,0.0,52.000000,26.000000,...,0.0,19.000000,17.000000,5.000000,57.000000,36.000000,44.000000,0.000000,33.000000,26.000000
2021-01-02,14.000000,17.000000,25.000000,77.000000,53.000000,41.000000,21.000000,0.0,59.000000,39.000000,...,0.0,28.000000,14.000000,6.000000,54.000000,51.000000,42.000000,1.000000,32.000000,30.000000
2021-01-03,13.000000,18.000000,25.000000,70.000000,52.000000,38.000000,26.000000,0.0,57.000000,47.000000,...,0.0,32.000000,13.000000,5.000000,51.000000,45.000000,75.000000,1.000000,47.000000,33.000000
2021-01-04,30.000000,25.000000,29.000000,79.000000,61.000000,69.000000,51.000000,0.0,73.000000,45.000000,...,0.0,25.000000,45.000000,4.000000,70.000000,44.000000,46.000000,2.000000,39.000000,56.000000
2021-01-05,29.000000,26.000000,31.000000,82.000000,68.000000,78.000000,60.000000,0.0,83.000000,37.000000,...,0.0,27.000000,44.000000,4.000000,58.000000,47.000000,50.000000,1.000000,53.000000,66.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-15,27.506775,20.402893,78.490104,127.613327,195.749222,498.996418,433.411618,NaN,149.859575,13.100188,...,NaN,6.149557,16.369440,5.061834,81.550799,124.783128,1209.159045,1.873363,221.773719,234.119008
2026-04-16,29.034929,20.402893,78.490104,124.777475,167.202461,476.314762,371.495673,NaN,172.179086,13.973534,...,NaN,6.764513,16.661751,4.814915,78.832439,183.015254,1577.163972,2.128822,242.565005,273.969052
2026-04-17,38.203854,22.103134,82.850666,124.777475,167.202461,884.584558,603.680468,NaN,178.556089,15.720226,...,NaN,8.609380,19.584865,7.160643,84.269159,128.942566,2260.601693,5.109172,270.286720,268.987796


In [16]:
### Create attention data features
attention_data = pd.read_csv('raw_attention_data.csv', index_col='date')
attention_data = attention_data.drop(columns=['Fed rate hike','memecoin'])  # remove columns with too many missing values

for kw in kws:
  if kw == 'Fed rate hike' or kw == 'memecoin':
    continue
  attention_data[kw + '_diff'] = attention_data[kw].diff()
  attention_data[kw+'_shock_10'] = (attention_data[kw]-attention_data[kw].rolling(window=10).mean()) / attention_data[kw].rolling(window=10).std()

attention_data = attention_data.dropna()
attention_data.to_csv('attention_data.csv')
attention_data

,stock market,stock market crash,inflation,interest rates,recession,oil prices,crude oil,economy,bitcoin price,crypto crash,...,electric vehicles_diff,electric vehicles_shock_10,AI stocks_diff,AI stocks_shock_10,tesla crash_diff,tesla crash_shock_10,battery technology_diff,battery technology_shock_10,lithium price_diff,lithium price_shock_10
date,,,,,,,,,,,,,,,,,,,,,
2021-01-10,15.000000,29.000000,26.000000,70.000000,56.000000,42.000000,29.000000,66.000000,44.000000,1.000000,...,2.000000,0.357938,15.000000,1.751993,-1.000000,0.241747,-6.000000,0.159384,-1.000000,0.378763
2021-01-11,32.000000,32.000000,31.000000,81.000000,70.000000,71.000000,55.000000,85.000000,70.000000,4.000000,...,6.000000,1.571240,-9.000000,0.855122,0.000000,0.106735,17.000000,1.645489,7.000000,0.614288
2021-01-12,31.000000,33.000000,33.000000,78.000000,68.000000,69.000000,59.000000,87.000000,55.000000,1.000000,...,3.000000,1.750014,1.000000,0.768610,-1.000000,-0.502244,2.000000,1.595723,18.000000,1.543675
2021-01-13,30.000000,29.000000,34.000000,78.000000,76.000000,72.000000,60.000000,84.000000,45.000000,1.000000,...,5.000000,1.922684,-28.000000,-1.311322,0.000000,-0.588348,-13.000000,0.035190,0.000000,1.577657
2021-01-14,30.000000,32.000000,35.000000,74.000000,68.000000,70.000000,58.000000,87.000000,43.000000,1.000000,...,-3.000000,1.170791,-1.000000,-1.285372,0.000000,-0.588348,0.000000,-0.156102,10.000000,1.851923
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-15,27.506775,20.402893,78.490104,127.613327,195.749222,498.996418,433.411618,149.859575,13.100188,0.906402,...,16.637750,-0.224818,105.144265,-0.239597,0.340611,-0.653552,6.930429,-0.363903,4.981255,-0.080602
2026-04-16,29.034929,20.402893,78.490104,124.777475,167.202461,476.314762,371.495673,172.179086,13.973534,1.087682,...,58.232126,1.189642,368.004927,0.553388,0.255459,-0.430880,20.791286,-0.057341,39.850044,0.557387
2026-04-17,38.203854,22.103134,82.850666,124.777475,167.202461,884.584558,603.680468,178.556089,15.720226,1.812803,...,-54.072689,0.306897,683.437721,2.384378,2.980351,2.340255,27.721715,1.886701,-4.981255,1.436876


### Allign Data

In [30]:
asset_data = pd.read_csv('asset_data.csv')
attention_data = pd.read_csv('attention_data.csv')
data = pd.concat([asset_data,attention_data],axis=1)
data = data.dropna()
data.to_csv('data.csv')
data

,Date,SPY_log_ret,SPY_log_ret_t+1,SPY_vol_10,SPY_vol_10_t+1,SPY_vol_change,SPY_shock_10,SPY_shock_10_t+1,BTC-USD_log_ret,BTC-USD_log_ret_t+1,...,electric vehicles_diff,electric vehicles_shock_10,AI stocks_diff,AI stocks_shock_10,tesla crash_diff,tesla crash_shock_10,battery technology_diff,battery technology_shock_10,lithium price_diff,lithium price_shock_10
0,2021-01-20,0.013744,0.000911,0.007772,0.007746,0.000757,1.768383,0.117644,-0.014579,-0.142528,...,2.000000,0.357938,15.000000,1.751993,-1.000000,0.241747,-6.000000,0.159384,-1.000000,0.378763
1,2021-01-21,0.000911,-0.003546,0.007746,0.006707,-0.000026,0.117644,-0.528734,-0.142528,0.068333,...,6.000000,1.571240,-9.000000,0.855122,0.000000,0.106735,17.000000,1.645489,7.000000,0.614288
2,2021-01-22,-0.003546,0.003936,0.006707,0.006593,-0.001039,-0.528734,0.597069,0.068333,-0.019562,...,3.000000,1.750014,1.000000,0.768610,-1.000000,-0.502244,2.000000,1.595723,18.000000,1.543675
3,2021-01-25,0.003936,-0.001562,0.006593,0.006116,-0.000114,0.597069,-0.255441,-0.019562,0.006266,...,5.000000,1.922684,-28.000000,-1.311322,0.000000,-0.588348,-13.000000,0.035190,0.000000,1.577657
4,2021-01-26,-0.001562,-0.024744,0.006116,0.010292,-0.000477,-0.255441,-2.404135,0.006266,-0.067874,...,-3.000000,1.170791,-1.000000,-1.285372,0.000000,-0.588348,0.000000,-0.156102,10.000000,1.851923
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1314,2026-04-15,0.007860,0.002454,0.007517,0.007673,-0.002588,1.045692,0.319860,0.008370,0.004629,...,-12.811396,-1.396643,-49.403305,-1.951673,-1.862237,-2.458357,-18.904243,-1.947492,-13.347635,-1.585751
1315,2026-04-16,0.002454,0.012013,0.007673,0.007520,0.000156,0.319860,1.597582,0.004629,0.025937,...,-4.270465,-1.690605,148.209916,-0.477153,1.513068,-0.164632,-6.301414,-1.813714,6.673817,-0.735389
1316,2026-04-17,0.012013,-0.002002,0.007520,0.008118,-0.000153,1.597582,-0.246586,0.025937,-0.016397,...,23.487559,1.693843,0.000000,-0.346071,-0.232780,-0.429573,12.602829,-0.727047,13.347635,0.805788
1317,2026-04-20,-0.002002,-0.006568,0.008118,0.009026,0.000598,-0.246586,-0.727741,-0.016397,0.006310,...,-12.811396,-0.273196,-49.403305,-0.742275,0.814729,0.672610,6.301414,-0.271808,0.000000,0.649671


## 4: Exploratory Data Analysis

In [24]:
data = pd.read_csv('data.csv')
kws_map = {
    'SPY_kws': SPY_kws,
    'BTC-USD_kws': BTC_kws,
    'TSLA_kws': TSLA_kws
}



# generate a 'plots' folder to store plot files
os.makedirs('plots',exist_ok=True)


# loop over tickers, keywords, and stats to create plots
for ticker in tickers:
    for kw in kws_map[ticker+'_kws']:
        if kw == 'Fed rate hike' or kw == 'memecoin':  # skip problematic columns
            continue

        fig,axes = plt.subplots(3,3,figsize=(12,10),sharex=True) # generate 3x3 subplot of for each ticker kw pair
        axes=axes.flatten()
        i=0

        for ticker_stat in ['_log_ret','_vol_10','_shock_10']:
            for kw_stat in ['','_diff','_shock_10']:
                ax = axes[i]

                x = (data[ticker+ticker_stat][::10] - data[ticker+ticker_stat][::10].min()) / (
                    data[ticker+ticker_stat][::10].max() - data[ticker+ticker_stat][::10].min()
                )

                y = (data[kw+kw_stat][::10] - data[kw+kw_stat][::10].min()) / (
                    data[kw+kw_stat][::10].max() - data[kw+kw_stat][::10].min()
                )

                ax.plot(data.index[::10], x, label=ticker+ticker_stat)
                ax.plot(data.index[::10], y, label=kw+kw_stat)

                # correlation coefficient
                corr = x.corr(y)

                # put in top-left corner
                ax.text(
                    0.02, 0.98,
                    f"corr={corr:.2f}",
                    transform=ax.transAxes,
                    ha='left',
                    va='top',
                    fontsize=9
                )

                ax.set_title(f'{ticker+ticker_stat} vs {kw+kw_stat}')
                ax.legend(fontsize=8)

                i += 1
        
        plt.tight_layout()
        path = 'plots/' + ticker + '_'+kw+'.png'
        plt.savefig(path,dpi=300)
        plt.close()

## 5: Modeling

#### Train-Test Split and Processing

In [31]:
train = data.loc[:'2025-12-31']
test = data.loc[:'2026-01-02']

#### Baseline Volatility Model (AR)

SPY (overall market)

In [232]:
y_train = train['SPY_shock_10_t+1']
y_test = test['SPY_shock_10_t+1']
X_train = pd.DataFrame()
X_test = pd.DataFrame()
X_train['lag0'] = train['SPY_shock_10']
X_test['lag0'] = test['SPY_shock_10']
X_train['lag1']=X_train['lag0'].shift(1)
X_test['lag1']=X_test['lag0'].shift(1)
X_train['lag2']=X_train['lag0'].shift(2)
X_test['lag2']=X_test['lag0'].shift(2)
X_train = X_train.dropna()
X_test = X_test.dropna()
y_train = y_train[X_train.index]
y_test = y_test[X_test.index]
X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

model = sm.OLS(y,X).fit()
print(model.summary())

predictions = model.predict(X_test)
SPY_base_mse = mean_squared_error(y_test, predictions)
SPY_base_mae = mean_absolute_error(y_test, predictions)
print(f'MSE: {SPY_base_mse}')
print(f'MAE: {SPY_base_mae}')

                             OLS Regression Results                             
Dep. Variable:     BTC-USD_shock_10_t+1   R-squared:                       0.003
Model:                              OLS   Adj. R-squared:                  0.001
Method:                   Least Squares   F-statistic:                     1.401
Date:                  Wed, 22 Apr 2026   Prob (F-statistic):              0.241
Time:                          22:39:58   Log-Likelihood:                -1871.8
No. Observations:                  1317   AIC:                             3752.
Df Residuals:                      1313   BIC:                             3772.
Df Model:                             3                                         
Covariance Type:              nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const               

BTC-USD

In [234]:
y_train = train['BTC-USD_shock_10_t+1']
y_test = test['BTC-USD_shock_10_t+1']
X_train = pd.DataFrame()
X_test = pd.DataFrame()
X_train['lag0'] = train['BTC-USD_shock_10']
X_test['lag0'] = test['BTC-USD_shock_10']
X_train['lag1']=X_train['lag0'].shift(1)
X_test['lag1']=X_test['lag0'].shift(1)
X_train['lag2']=X_train['lag0'].shift(2)
X_test['lag2']=X_test['lag0'].shift(2)
X_train = X_train.dropna()
X_test = X_test.dropna()
y_train = y_train[X_train.index]
y_test = y_test[X_test.index]
X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

model = sm.OLS(y,X).fit()
print(model.summary())

predictions = model.predict(X_test)
BTC_base_mse = mean_squared_error(y_test, predictions)
BTC_base_mae = mean_absolute_error(y_test, predictions)
print(f'MSE: {BTC_base_mse}')
print(f'MAE: {BTC_base_mae}')

                             OLS Regression Results                             
Dep. Variable:     BTC-USD_shock_10_t+1   R-squared:                       0.003
Model:                              OLS   Adj. R-squared:                  0.001
Method:                   Least Squares   F-statistic:                     1.401
Date:                  Wed, 22 Apr 2026   Prob (F-statistic):              0.241
Time:                          22:43:30   Log-Likelihood:                -1871.8
No. Observations:                  1317   AIC:                             3752.
Df Residuals:                      1313   BIC:                             3772.
Df Model:                             3                                         
Covariance Type:              nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const               

TSLA

In [236]:
y_train = train['TSLA_shock_10_t+1']
y_test = test['TSLA_shock_10_t+1']
X_train = pd.DataFrame()
X_test = pd.DataFrame()
X_train['lag0'] = train['TSLA_shock_10']
X_test['lag0'] = test['TSLA_shock_10']
X_train['lag1']=X_train['lag0'].shift(1)
X_test['lag1']=X_test['lag0'].shift(1)
X_train['lag2']=X_train['lag0'].shift(2)
X_test['lag2']=X_test['lag0'].shift(2)
X_train = X_train.dropna()
X_test = X_test.dropna()
y_train = y_train[X_train.index]
y_test = y_test[X_test.index]
X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

model = sm.OLS(y,X).fit()
print(model.summary())

predictions = model.predict(X_test)
TSLA_base_mse = mean_squared_error(y_test, predictions)
TSLA_base_mae = mean_absolute_error(y_test, predictions)
print(f'MSE: {TSLA_base_mse}')
print(f'MAE: {TSLA_base_mae}')

                             OLS Regression Results                             
Dep. Variable:     BTC-USD_shock_10_t+1   R-squared:                       0.003
Model:                              OLS   Adj. R-squared:                  0.001
Method:                   Least Squares   F-statistic:                     1.401
Date:                  Wed, 22 Apr 2026   Prob (F-statistic):              0.241
Time:                          22:45:21   Log-Likelihood:                -1871.8
No. Observations:                  1317   AIC:                             3752.
Df Residuals:                      1313   BIC:                             3772.
Df Model:                             3                                         
Covariance Type:              nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const               

#### Augmented Volatility Model (with attention metrics)

SPY (overall market)

In [ ]:
# analyze performance for each metric and find best improvements
SPY_best_metrics = []
y_train = train['SPY_shock_10_t+1']
y_test = test['SPY_shock_10_t+1']
for kw in [i for i in SPY_kws if i!='Fed rate hike']:
    for stat in ['','_diff','_shock_10']:

        y_train = pd.DataFrame()
        y_test = pd.DataFrame()
        y_train = train['SPY_shock_10_t+1']
        y_test = test['SPY_shock_10_t+1']
        X_train = pd.DataFrame()
        X_test = pd.DataFrame()
        X_train[kw+stat]=train[kw+stat]
        X_test[kw+stat]=test[kw+stat]
        X_train['lag0'] = train['SPY_shock_10']
        X_test['lag0'] = test['SPY_shock_10']
        X_train['lag1']=X_train['lag0'].shift(1)
        X_test['lag1']=X_test['lag0'].shift(1)
        X_train['lag2']=X_train['lag0'].shift(2)
        X_test['lag2']=X_test['lag0'].shift(2)
        X_train = X_train.dropna()
        X_test = X_test.dropna()
        y_train = y_train[X_train.index]
        y_test = y_test[X_test.index]
        X_train = sm.add_constant(X_train)
        X_test = sm.add_constant(X_test)

        model = sm.OLS(y_train,X_train).fit()
        predictions = model.predict(X_test)
        mse = mean_squared_error(y_test, predictions)
        if mse<SPY_base_mse:  # only analyze kw+stat pairs with improvement over baseline
            print(model.summary())
            print(f'MSE: {mse}')
            SPY_best_metrics.append((kw+stat,mse))

                            OLS Regression Results                            
Dep. Variable:       SPY_shock_10_t+1   R-squared:                       0.012
Model:                            OLS   Adj. R-squared:                 -0.008
Method:                 Least Squares   F-statistic:                    0.6055
Date:                Wed, 22 Apr 2026   Prob (F-statistic):              0.659
Time:                        22:41:40   Log-Likelihood:                -287.64
No. Observations:                 201   AIC:                             585.3
Df Residuals:                     196   BIC:                             601.8
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.1457      0.164      0.888   

In [249]:
# create model with 5 best metrics
SPY_t5 = sorted(SPY_best_metrics, key=lambda x: x[1])[:5]
for metric in SPY_t5:
    print(f'Metric: {metric[0]}   MSE: {metric[1]}')
SPY_t5 = [x[0] for x in SPY_t5]
y_train = train['SPY_shock_10_t+1']
y_test = test['SPY_shock_10_t+1']
X_train = pd.DataFrame()
X_test = pd.DataFrame()
X_train[SPY_t5]=train[SPY_t5]
X_test[SPY_t5]=test[SPY_t5]
X_train['lag0'] = train['SPY_shock_10']
X_test['lag0'] = test['SPY_shock_10']
X_train['lag1']=X_train['lag0'].shift(1)
X_test['lag1']=X_test['lag0'].shift(1)
X_train['lag2']=X_train['lag0'].shift(2)
X_test['lag2']=X_test['lag0'].shift(2)
X_train = X_train.dropna()
X_test = X_test.dropna()
y_train = y_train[X_train.index]
y_test = y_test[X_test.index]
X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

model = sm.OLS(y_train,X_train).fit()
predictions = model.predict(X_test)
mse = mean_squared_error(y_test, predictions)
print(model.summary())
print(f'MSE: {mse}')
print(f'MAE: {mean_absolute_error(y_test,predictions)}')

Metric: crude oil_diff   MSE: 1.0165508007783297
Metric: interest rates_shock_10   MSE: 1.0177028024535557
Metric: oil prices_shock_10   MSE: 1.0182724835273718
Metric: inflation   MSE: 1.0200374311904816
Metric: economy_shock_10   MSE: 1.0206951826472919
                            OLS Regression Results                            
Dep. Variable:       SPY_shock_10_t+1   R-squared:                       0.048
Model:                            OLS   Adj. R-squared:                  0.009
Method:                 Least Squares   F-statistic:                     1.221
Date:                Wed, 22 Apr 2026   Prob (F-statistic):              0.289
Time:                        23:15:21   Log-Likelihood:                -283.88
No. Observations:                 201   AIC:                             585.8
Df Residuals:                     192   BIC:                             615.5
Df Model:                           8                                         
Covariance Type:            nonro

BTC-USD

In [220]:
# analyze performance for each metric and find best improvements
BTC_best_metrics = []
y_train = train['BTC-USD_shock_10_t+1']
y_test = test['BTC-USD_shock_10_t+1']
for kw in [i for i in BTC_kws if i!='memecoin']:
    for stat in ['','_diff','_shock_10']:

        y_train = pd.DataFrame()
        y_test = pd.DataFrame()
        y_train = train['BTC-USD_shock_10_t+1']
        y_test = test['BTC-USD_shock_10_t+1']
        X_train = pd.DataFrame()
        X_test = pd.DataFrame()
        X_train[kw+stat]=train[kw+stat]
        X_test[kw+stat]=test[kw+stat]
        X_train['lag0'] = train['BTC-USD_shock_10']
        X_test['lag0'] = test['BTC-USD_shock_10']
        X_train['lag1']=X_train['lag0'].shift(1)
        X_test['lag1']=X_test['lag0'].shift(1)
        X_train['lag2']=X_train['lag0'].shift(2)
        X_test['lag2']=X_test['lag0'].shift(2)
        X_train = X_train.dropna()
        X_test = X_test.dropna()
        y_train = y_train[X_train.index]
        y_test = y_test[X_test.index]
        X_train = sm.add_constant(X_train)
        X_test = sm.add_constant(X_test)

        model = sm.OLS(y_train,X_train).fit()
        predictions = model.predict(X_test)
        mse = mean_squared_error(y_test, predictions)
        if mse<BTC_base_mse:  # only analyze kw+stat pairs with improvement over baseline
            print(model.summary())
            print(f'MSE: {mse}')
            BTC_best_metrics.append((kw+stat,mse))

                             OLS Regression Results                             
Dep. Variable:     BTC-USD_shock_10_t+1   R-squared:                       0.011
Model:                              OLS   Adj. R-squared:                 -0.009
Method:                   Least Squares   F-statistic:                    0.5332
Date:                  Wed, 22 Apr 2026   Prob (F-statistic):              0.711
Time:                          22:28:48   Log-Likelihood:                -279.97
No. Observations:                   201   AIC:                             569.9
Df Residuals:                       196   BIC:                             586.5
Df Model:                             4                                         
Covariance Type:              nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -0.0663  

In [250]:
# create model with 5 best metrics
BTC_t5 = sorted(BTC_best_metrics, key=lambda x: x[1])[:5]
for metric in BTC_t5:
    print(f'Metric: {metric[0]}   MSE: {metric[1]}')
BTC_t5 = [x[0] for x in BTC_t5]
y_train = train['BTC-USD_shock_10_t+1']
y_test = test['BTC-USD_shock_10_t+1']
X_train = pd.DataFrame()
X_test = pd.DataFrame()
X_train[BTC_t5]=train[BTC_t5]
X_test[BTC_t5]=test[BTC_t5]
X_train['lag0'] = train['BTC-USD_shock_10']
X_test['lag0'] = test['BTC-USD_shock_10']
X_train['lag1']=X_train['lag0'].shift(1)
X_test['lag1']=X_test['lag0'].shift(1)
X_train['lag2']=X_train['lag0'].shift(2)
X_test['lag2']=X_test['lag0'].shift(2)
X_train = X_train.dropna()
X_test = X_test.dropna()
y_train = y_train[X_train.index]
y_test = y_test[X_test.index]
X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

model = sm.OLS(y_train,X_train).fit()
predictions = model.predict(X_test)
mse = mean_squared_error(y_test, predictions)
print(model.summary())
print(f'MSE: {mse}')
print(f'MAE: {mean_absolute_error(y_test,predictions)}')

Metric: altcoin   MSE: 0.9217497651865896
Metric: altcoin_shock_10   MSE: 0.9255393357090946
Metric: crypto_shock_10   MSE: 0.9283476939363855
Metric: risk off_shock_10   MSE: 0.933010781729559
Metric: bitcoin mining_shock_10   MSE: 0.934969335685629
                             OLS Regression Results                             
Dep. Variable:     BTC-USD_shock_10_t+1   R-squared:                       0.069
Model:                              OLS   Adj. R-squared:                  0.030
Method:                   Least Squares   F-statistic:                     1.769
Date:                  Wed, 22 Apr 2026   Prob (F-statistic):             0.0853
Time:                          23:15:47   Log-Likelihood:                -273.91
No. Observations:                   201   AIC:                             565.8
Df Residuals:                       192   BIC:                             595.5
Df Model:                             8                                         
Covariance Type:    

TSLA

In [241]:
# analyze performance for each metric and find best improvements
TSLA_best_metrics = []
y_train = train['TSLA_shock_10_t+1']
y_test = test['TSLA_shock_10_t+1']
for kw in [i for i in TSLA_kws if i!='memecoin']:
    for stat in ['','_diff','_shock_10']:

        y_train = pd.DataFrame()
        y_test = pd.DataFrame()
        y_train = train['TSLA_shock_10_t+1']
        y_test = test['TSLA_shock_10_t+1']
        X_train = pd.DataFrame()
        X_test = pd.DataFrame()
        X_train[kw+stat]=train[kw+stat]
        X_test[kw+stat]=test[kw+stat]
        X_train['lag0'] = train['TSLA_shock_10']
        X_test['lag0'] = test['TSLA_shock_10']
        X_train['lag1']=X_train['lag0'].shift(1)
        X_test['lag1']=X_test['lag0'].shift(1)
        X_train['lag2']=X_train['lag0'].shift(2)
        X_test['lag2']=X_test['lag0'].shift(2)
        X_train = X_train.dropna()
        X_test = X_test.dropna()
        y_train = y_train[X_train.index]
        y_test = y_test[X_test.index]
        X_train = sm.add_constant(X_train)
        X_test = sm.add_constant(X_test)

        model = sm.OLS(y_train,X_train).fit()
        predictions = model.predict(X_test)
        mse = mean_squared_error(y_test, predictions)
        if mse<TSLA_base_mse:  # only analyze kw+stat pairs with improvement over baseline
            print(model.summary())
            print(f'MSE: {mse}')
            TSLA_best_metrics.append((kw+stat,mse))

                            OLS Regression Results                            
Dep. Variable:      TSLA_shock_10_t+1   R-squared:                       0.041
Model:                            OLS   Adj. R-squared:                  0.022
Method:                 Least Squares   F-statistic:                     2.103
Date:                Wed, 22 Apr 2026   Prob (F-statistic):             0.0819
Time:                        22:51:42   Log-Likelihood:                -288.92
No. Observations:                 201   AIC:                             587.8
Df Residuals:                     196   BIC:                             604.4
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.3624      0.144      2.517      

In [251]:
# create model with 5 best metrics
TSLA_t5 = sorted(TSLA_best_metrics, key=lambda x: x[1])[:5]
for metric in TSLA_t5:
    print(f'Metric: {metric[0]}   MSE: {metric[1]}')
TSLA_t5 = [x[0] for x in TSLA_t5]
y_train = train['TSLA_shock_10_t+1']
y_test = test['TSLA_shock_10_t+1']
X_train = pd.DataFrame()
X_test = pd.DataFrame()
X_train[TSLA_t5]=train[TSLA_t5]
X_test[TSLA_t5]=test[TSLA_t5]
X_train['lag0'] = train['TSLA_shock_10']
X_test['lag0'] = test['TSLA_shock_10']
X_train['lag1']=X_train['lag0'].shift(1)
X_test['lag1']=X_test['lag0'].shift(1)
X_train['lag2']=X_train['lag0'].shift(2)
X_test['lag2']=X_test['lag0'].shift(2)
X_train = X_train.dropna()
X_test = X_test.dropna()
y_train = y_train[X_train.index]
y_test = y_test[X_test.index]
X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

model = sm.OLS(y_train,X_train).fit()
predictions = model.predict(X_test)
mse = mean_squared_error(y_test, predictions)
print(model.summary())
print(f'MSE: {mse}')
print(f'MAE: {mean_absolute_error(y_test,predictions)}')

Metric: battery technology   MSE: 1.0281727441497939
Metric: tesla stock   MSE: 1.0376471513054057
Metric: battery technology_shock_10   MSE: 1.0456790652111292
Metric: tesla crash_shock_10   MSE: 1.0475095863038517
Metric: self driving car   MSE: 1.0481984364082513
                            OLS Regression Results                            
Dep. Variable:      TSLA_shock_10_t+1   R-squared:                       0.080
Model:                            OLS   Adj. R-squared:                  0.041
Method:                 Least Squares   F-statistic:                     2.080
Date:                Wed, 22 Apr 2026   Prob (F-statistic):             0.0395
Time:                        23:16:00   Log-Likelihood:                -284.79
No. Observations:                 201   AIC:                             587.6
Df Residuals:                     192   BIC:                             617.3
Df Model:                           8                                         
Covariance Type:      